# Week 4 Quiz — Optimisation, Delta Lake & Unity Catalog

**Topics:** Catalyst/Tungsten, AQE, Dynamic Partition Pruning, Delta Lake (ACID, MERGE, VACUUM, Z-ORDER, CDF, time travel), Unity Catalog, Spark Connect

**18 questions.** Run each answer cell to reveal the correct answer and explanation.

---

## Q1

An organisation needs an optimised storage layer that supports **ACID transactions** and **schema enforcement**. Which technology should they use?

- A. Unity Catalog
- B. Traditional data lake (raw cloud object storage)
- **C. Delta Lake**
- D. Cloud File Storage (raw S3/ADLS/GCS)

In [ ]:
answer = "C"
explanation = (
    "Delta Lake is an open-source storage layer that adds ACID transactions, scalable metadata, "
    "and schema enforcement/evolution on top of cloud object storage. It is the definitive answer "
    "for lakehouse ACID guarantees. "
    "Unity Catalog is a governance layer (access control, lineage) — not a storage format. "
    "Traditional data lakes lack ACID transactions (one of the problems Delta Lake was built to fix). "
    "Raw cloud storage (S3/ADLS/GCS) has no ACID or schema enforcement natively."
)
print(f"Q1 → {answer}  |  {explanation}")

## Q2

A data architect needs to create an empty table with a defined schema **regardless of whether a table with that name already exists**. Which DDL command achieves this?

- **A. `CREATE OR REPLACE TABLE table_name (employeeid STRING, startDate DATE, avgRating FLOAT) USING DELTA`**
- B. `CREATE OR REPLACE TABLE table_name WITH COLUMNS (employeeid STRING, startDate DATE, avgRating FLOAT) USING DELTA`
- C. `CREATE TABLE IF NOT EXISTS table_name (employeeid STRING, startDate DATE, avgRating FLOAT)`
- D. `CREATE TABLE table_name AS SELECT employeeid STRING, startDate DATE, avgRating FLOAT`

In [ ]:
answer = "A"
explanation = (
    "CREATE OR REPLACE TABLE drops the existing table (if any) and recreates it fresh — it works "
    "regardless of prior existence. 'WITH COLUMNS' (B) is not valid DDL syntax. "
    "CREATE TABLE IF NOT EXISTS (C) does nothing if the table already exists — the opposite of what "
    "is needed here. CREATE TABLE AS SELECT (D) creates a table from query results, not from an "
    "explicit column definition, and the syntax shown is invalid SQL."
)
print(f"Q2 → {answer}  |  {explanation}")

## Q3

Rows meeting a query filter condition are **sparsely distributed** across many data files in a Delta table. File size tuning has already been done. Which optimisation technique should be applied next?

- A. Data skipping
- **B. Z-Ordering**
- C. Bin-packing (OPTIMIZE without ZORDER)
- D. Write the data as Parquet without Delta
- E. Tune the file size again

In [ ]:
answer = "B"
explanation = (
    "Z-Ordering (OPTIMIZE table ZORDER BY col) co-locates related data within the same files by "
    "sorting on the specified column(s). When rows matching a filter are sparsely scattered, "
    "Z-Ordering clusters them together, tightening min/max statistics per file so Delta's data "
    "skipping can skip irrelevant files entirely. Data skipping (A) is the *result* of Z-Ordering, "
    "not an action. Bin-packing (C) fixes small files — already done. Writing plain Parquet (D) "
    "loses Delta features. File size tuning (E) is already exhausted."
)
print(f"Q3 → {answer}  |  {explanation}")

## Q4

A data engineer needs to create a database named `customer360` at `/customer/customer360/` and is unsure if a colleague already created it. Which command should they run?

- A. `CREATE DATABASE customer360 LOCATION '/customer/customer360';`
- B. `CREATE DATABASE IF NOT EXISTS customer360 LOCATION '/customer/customer360';`
- **C. `CREATE DATABASE IF NOT EXISTS customer360 LOCATION '/customer/customer360';`**
- D. `CREATE DATABASE IF NOT EXISTS customer360 DELTA LOCATION '/customer/customer360';`
- E. `CREATE DATABASE customer360 DELTA LOCATION '/customer/customer360';`

In [ ]:
answer = "C"
explanation = (
    "CREATE DATABASE IF NOT EXISTS is idempotent — it creates the database if it does not exist "
    "and silently does nothing if it already exists. The LOCATION clause specifies where the "
    "database directory will be created on cloud storage. "
    "Option A omits IF NOT EXISTS and will error if the database already exists. "
    "Options D and E use 'DELTA LOCATION' which is not valid SQL syntax for CREATE DATABASE — "
    "the correct keyword is simply LOCATION. (B and C appear identical in the question; C is marked.)"
)
print(f"Q4 → {answer}  |  {explanation}")

## Q5

A data engineer needs to create a project home directory and **maintain a specific path** in an external location. Which table type is appropriate?

- **A. An external table with a LOCATION clause pointing to the specific external path**
- B. An external table where the schema has a managed location pointing to the external path
- C. A managed table where the catalog has a managed location pointing to the external path
- D. A managed table with a LOCATION clause pointing to the external path

In [ ]:
answer = "A"
explanation = (
    "External tables are created with a LOCATION clause that explicitly points to a path on cloud "
    "storage that the engineer controls. Dropping the table removes only the metadata — data at "
    "the location is preserved. This is the correct choice when path control is required. "
    "Managed tables have their storage path controlled by the platform (Databricks/Unity Catalog); "
    "you cannot specify a custom LOCATION for a managed table in Unity Catalog."
)
print(f"Q5 → {answer}  |  {explanation}")

## Q6

Which command can be used to **write data into a Delta table while avoiding duplicate records**?

- A. DROP
- B. IGNORE
- **C. MERGE**
- D. APPEND
- E. INSERT

In [ ]:
answer = "C"
explanation = (
    "MERGE INTO (upsert) matches incoming source rows against the target table using a join condition. "
    "You can configure WHEN MATCHED (update existing) and WHEN NOT MATCHED (insert only new). "
    "This prevents duplicate records because rows that already exist are updated, not re-inserted. "
    "APPEND unconditionally adds all rows. DROP deletes the entire table. "
    "INSERT has no built-in deduplication logic by itself."
)
print(f"Q6 → {answer}  |  {explanation}")

## Q7

After running `DROP TABLE IF EXISTS my_table`, a data engineer notices the **data files still exist** but the metadata was deleted. Why?

- A. The table's data was larger than 10 GB
- B. The table's data was smaller than 10 GB
- **C. The table was an external table**
- D. The table did not have a LOCATION clause
- E. The table was a managed table

In [ ]:
answer = "C"
explanation = (
    "External tables store data at a user-specified location that is considered 'externally owned'. "
    "When you DROP an external table, Spark removes only the metastore metadata entry — the data "
    "files at the LOCATION path are preserved by design. "
    "Managed tables (E) do the opposite: DROP TABLE removes BOTH metadata AND data files. "
    "Table size has no effect on DROP TABLE behavior. The absence of a LOCATION is what "
    "makes a table managed (no LOCATION = managed)."
)
print(f"Q7 → {answer}  |  {explanation}")

## Q8

After `DROP TABLE IF EXISTS my_table`, both the data files **and** metadata files are deleted. What explains this?

- **A. The table was a managed table**
- B. The table's data was smaller than 10 GB
- C. The table had no LOCATION clause
- D. The table was an external table
- E. The table's data was larger than 10 GB

In [ ]:
answer = "A"
explanation = (
    "A managed table is fully owned by Spark/Databricks — both the metadata (metastore registration) "
    "and the data files (stored in the default warehouse path) are managed by the platform. "
    "Dropping a managed table deletes BOTH. This is the core behavioral distinction from external "
    "tables: external = metadata only deleted; managed = metadata + data deleted. "
    "Table size is irrelevant to this behavior."
)
print(f"Q8 → {answer}  |  {explanation}")

## Q9

Which **file format** does Delta Lake use to physically store table data?

- A. CSV
- **B. Parquet**
- C. JSON
- D. Delta (proprietary format)

In [ ]:
answer = "B"
explanation = (
    "Delta Lake stores actual table data in Parquet files. The 'Delta' format adds a transaction log "
    "directory (_delta_log/) on top of those Parquet files containing JSON commit files and periodic "
    "Parquet checkpoint files. JSON is used only for the commit log entries, not for data. "
    "CSV lacks columnar efficiency and schema embedding. There is no proprietary 'Delta' file format — "
    "the data is plain Parquet readable by any Parquet-compatible tool."
)
print(f"Q9 → {answer}  |  {explanation}")

## Q10

A senior data engineer runs:
```sql
CREATE TABLE customersPerCountry AS
SELECT country, COUNT(*) AS customer_count
FROM customercustomization
GROUP BY country;
```
A junior engineer asks why no schema was declared. Which response is correct?

- **A. CTAS statements derive the schema from the SELECT query output**
- B. CTAS statements infer schema by parsing the raw data files
- C. CTAS statements result in tables where schemas are optional
- D. CTAS statements assign all columns the STRING type
- E. CTAS tables do not support schemas

In [ ]:
answer = "A"
explanation = (
    "CREATE TABLE AS SELECT (CTAS) automatically derives the new table's schema from the SELECT "
    "query result: column names come from the SELECT aliases/expressions, and data types are "
    "inferred from the query output (e.g. COUNT(*) → BIGINT, country → STRING). "
    "No explicit schema declaration is required or needed. This is not schema inference from raw "
    "files (B) — CTAS uses the SQL engine's typed output directly. CTAS tables always have a schema."
)
print(f"Q10 → {answer}  |  {explanation}")

## Q11

A colleague suggests overwriting a table instead of dropping and recreating it. Which reason for using **INSERT OVERWRITE / write.mode('overwrite')** is correct?

- A. Overwriting is efficient because no files need to be deleted
- B. Overwriting results in a clean table history for audit purposes
- C. Overwriting maintains the old version of the table for Time Travel
- **D. Overwriting is an atomic operation and will never leave the table in a partially written state**
- E. Overwriting allows concurrent queries to complete freely while in progress

In [ ]:
answer = "D"
explanation = (
    "Delta Lake overwrites are atomic: the new data is written to new files and the transaction log "
    "is updated in a single commit. If the write fails mid-way, the old data remains intact and "
    "visible — the table is never in a partially written state. "
    "Dropping and recreating is NOT atomic (window of time where the table doesn't exist). "
    "Option C (time travel) is also a valid benefit, but D is the most precise and defensible answer. "
    "Option B is backwards: overwriting preserves history; dropping destroys it."
)
print(f"Q11 → {answer}  |  {explanation}")

## Q12

A data analyst runs `SELECT district, count(1) FROM store_sales_20220101 GROUP BY district`. The table name contains the date. A data engineering team wants to **automate** this so the date in the table name is always the current date. Which approach should they use?

- **A. Rewrite the query in PySpark and use Python's string formatting (f-strings) to inject the current date into the table name**
- B. Manually update the date in the table name each day
- C. Ask the analyst to rewrite the query to run less frequently
- D. Replace the string-formatted date with a timestamp-formatted date in the table name

In [ ]:
answer = "A"
explanation = (
    "Python f-strings or .format() allow dynamic table name injection at runtime: "
    "table_name = f'store_sales_{datetime.now().strftime(\"%Y%m%d")}' "
    "then spark.sql(f'SELECT district, count(1) FROM {table_name} GROUP BY district'). "
    "This is fully automated — schedule via Databricks Jobs and the table name updates daily. "
    "Option B requires manual intervention every run, breaking automation. "
    "Option C reduces frequency but doesn't automate anything. "
    "Option D changes the naming scheme without solving the automation problem."
)
print(f"Q12 → {answer}  |  {explanation}")

## Q13

In which scenario should a data engineer use **INSERT OVERWRITE** instead of **INSERT INTO**?

- A. When the data location needs to change
- B. When the target table is an external table
- C. When the source table can be deleted after the operation
- **D. When the target table cannot contain duplicate records**
- E. When the source is not a Delta table

In [ ]:
answer = "D"
explanation = (
    "INSERT OVERWRITE completely replaces the contents of the target table with the new data in "
    "one atomic operation, ensuring no duplicates can exist from prior loads. "
    "INSERT INTO appends new rows to existing rows — if run twice with the same data, it creates "
    "duplicates. "
    "INSERT OVERWRITE does not change the table's storage location (A is wrong). "
    "Both commands work on managed and external tables (B is wrong). "
    "The source table type is irrelevant to the choice (E is wrong)."
)
print(f"Q13 → {answer}  |  {explanation}")

## Q14

An e-commerce transactions table is partitioned by `purchase_date`. Most queries filter on `customer_id` (high-cardinality), causing full partition scans and slow performance. How should the data engineer optimise the data layout?

- **A. Run `OPTIMIZE table ZORDER BY (customer_id)` to co-locate data by customer within each date partition**
- B. Repartition the table by `customer_id` instead of `purchase_date`
- C. Enable Delta caching on the cluster
- D. Run `OPTIMIZE table ZORDER BY (customer_id, purchase_date)`

In [ ]:
answer = "A"
explanation = (
    "Z-Ordering by customer_id co-locates rows with the same customer within each partition's files. "
    "Delta's data skipping reads min/max statistics per file and skips files that can't contain "
    "the queried customer_id, drastically reducing I/O. This preserves the existing purchase_date "
    "partitioning. "
    "Option B (repartition by customer_id) would create thousands of small partitions (one per customer) "
    "— the small files problem. "
    "Delta caching (C) speeds up repeated reads but doesn't fix layout. "
    "Z-Ordering on both (D) is less effective since purchase_date is already handled by partitioning."
)
print(f"Q14 → {answer}  |  {explanation}")

## Q15

Which three runtime optimisations does **Adaptive Query Execution (AQE)** perform in Spark?

- A. Predicate pushdown, broadcast hash joins, and Z-ordering
- **B. Post-shuffle partition coalescing, converting SortMergeJoin to BroadcastHashJoin at runtime, and skew join splitting**
- C. Repartitioning data, caching intermediate results, and dynamic partition pruning
- D. Column pruning, partition pruning, and join reordering at plan time

In [ ]:
answer = "B"
explanation = (
    "AQE makes three key runtime decisions after each shuffle stage using actual data statistics: "
    "1) Post-shuffle coalescing — merges many small shuffle partitions into fewer larger ones. "
    "2) SMJ→BHJ switch — if one side of a join turns out small enough after the shuffle, "
    "AQE switches from SortMergeJoin to the cheaper BroadcastHashJoin at runtime. "
    "3) Skew join splitting — detects skewed partitions and splits them into sub-tasks to "
    "avoid one slow task holding up the stage. "
    "Predicate pushdown and column pruning happen at plan time (Catalyst), not via AQE."
)
print(f"Q15 → {answer}  |  {explanation}")

## Q16

A data engineer wants to read the original state of a Delta table at `/data/delta` before any modifications (version 0). Which code is correct?

- A. `spark.read.format("delta").option("timestampAsOf", "0").load("/data/delta")`
- B. `spark.read.format("delta").option("version", 0).load("/data/delta")`
- **C. `spark.read.format("delta").option("versionAsOf", 0).load("/data/delta")`**
- D. `spark.read.format("delta").load("/data/delta").version(0)`

In [ ]:
answer = "C"
explanation = (
    "Delta Lake time travel uses exactly two option keys: "
    "'versionAsOf' — read a specific integer version number (0 = initial write). "
    "'timestampAsOf' — read the state as of a timestamp string (e.g. '2024-01-01'). "
    "Option A is wrong: timestampAsOf takes a timestamp string, not '0'. "
    "Option B uses 'version' — not the correct key name (must be 'versionAsOf'). "
    "Option D is invented syntax — there is no .version() method on DataFrameReader."
)
print(f"Q16 → {answer}  |  {explanation}")

## Q17

For **Dynamic Partition Pruning (DPP)** to be effective, which conditions must be true?

- A. The fact table must be fully cached in memory before the join
- B. AQE must be disabled for DPP to activate
- **C. The fact table must be partitioned on the join key, and the dimension table must be small enough to broadcast**
- D. Both tables must be stored as Delta Lake tables

In [ ]:
answer = "C"
explanation = (
    "DPP works by injecting the result of the dimension table filter as a runtime filter "
    "into the fact table scan. Two conditions are required: "
    "1) The fact table must be PARTITIONED on the join key — so Spark can prune entire "
    "partitions at scan time based on the injected filter. "
    "2) The dimension table must be broadcastable — Spark builds a bloom filter / "
    "in-list from the broadcasted dimension and pushes it into the fact table partition scan. "
    "DPP is enabled by spark.sql.optimizer.dynamicPartitionPruning.enabled=true (default true). "
    "AQE and DPP are independent; neither table needs to be Delta-specific."
)
print(f"Q17 → {answer}  |  {explanation}")

## Q18

A data engineer is using **Spark Connect** (serverless compute) in Databricks. Which of the following operations will **fail**?

- A. `spark.sql("SELECT * FROM my_table")`
- B. `spark.read.parquet("/data/path")`
- **C. `spark.sparkContext.accumulator(0)`**
- D. `df.groupBy("dept").agg(count("*"))`

In [ ]:
answer = "C"
explanation = (
    "Spark Connect (used in Databricks serverless) is a thin gRPC client that connects to a "
    "remote Spark server. The local Python process has NO direct access to the JVM/SparkContext, "
    "so spark.sparkContext raises an exception. This also means RDD operations, accumulators, "
    "and sc.broadcast() are unavailable — use DataFrame/SQL equivalents instead. "
    "DataFrame API operations (A, B, D) all work normally via the Spark Connect protocol since "
    "they are translated to gRPC plan protobuf messages and executed on the server."
)
print(f"Q18 → {answer}  |  {explanation}")